In [9]:
%%writefile config.py
from datetime import date
import numpy as np

NUM_PEOPLE = 2000
NUM_DAYS = 7
START_DATE = date(2024, 1, 1)
RNG = np.random.default_rng(7)

DEMOGRAPHIC_COLS = [
    "completed_8_weeks",
    "age",
    "biological_sex",
    "identifying_gender",
    "phone",
    "mental_health_diagnosis",
    "disorder_name___10",
    "disorder_name___1",
    "depression_score",
]

Overwriting config.py


In [10]:
%%writefile geo.py
import numpy as np


def haversine_meters(lat1, lon1, lat2, lon2):
    r = 6_371_000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

Overwriting geo.py


In [11]:
%%writefile personas.py
import pandas as pd

from config import NUM_PEOPLE, RNG


def make_personas():
    """Fixed per-participant traits that drive their simulated daily routine."""
    return pd.DataFrame({
        "chronotype_offset": RNG.normal(0, 1.0, NUM_PEOPLE),     # hrs, neg = early bird
        "activity_level": RNG.beta(2, 2, NUM_PEOPLE),            # 0-1
        "mobility_level": RNG.beta(2, 2, NUM_PEOPLE),            # 0-1, homebody -> roamer
        "screen_habit": RNG.beta(2, 2, NUM_PEOPLE),              # 0-1
        "home_lat": 40.0 + RNG.uniform(-0.15, 0.15, NUM_PEOPLE),
        "home_lon": -74.0 + RNG.uniform(-0.15, 0.15, NUM_PEOPLE),
    })


def make_demographics():
    completed_8_weeks = RNG.choice([0, 1], size=NUM_PEOPLE, p=[0.2, 0.8])
    age = RNG.integers(18, 66, size=NUM_PEOPLE)
    biological_sex = RNG.choice(["Male", "Female"], size=NUM_PEOPLE)
    identifying_gender = RNG.choice(
        ["Man", "Woman", "Non-binary", "Prefer not to say"],
        size=NUM_PEOPLE, p=[0.46, 0.46, 0.05, 0.03],
    )
    phone = RNG.choice(["iOS", "Android"], size=NUM_PEOPLE, p=[0.55, 0.45])
    mental_health_diagnosis = RNG.choice([0, 1], size=NUM_PEOPLE, p=[0.6, 0.4])

    disorder_name___1 = (mental_health_diagnosis == 1) & (RNG.random(NUM_PEOPLE) < 0.5)
    disorder_name___10 = (mental_health_diagnosis == 1) & (RNG.random(NUM_PEOPLE) < 0.3)

    # 1-60 scale; diagnosed participants skew higher
    depression_score = pd.Series(
        RNG.integers(1, 25, size=NUM_PEOPLE)
    ).where(mental_health_diagnosis == 0, RNG.integers(20, 61, size=NUM_PEOPLE))

    return pd.DataFrame({
        "completed_8_weeks": completed_8_weeks,
        "age": age,
        "biological_sex": biological_sex,
        "identifying_gender": identifying_gender,
        "phone": phone,
        "mental_health_diagnosis": mental_health_diagnosis,
        "disorder_name___10": disorder_name___10.astype(int),
        "disorder_name___1": disorder_name___1.astype(int),
        "depression_score": depression_score.to_numpy(),
    })

Overwriting personas.py


In [12]:
%%writefile gps_features.py
import numpy as np

from config import RNG
from geo import haversine_meters

GPS_COLS = [
    "gps_total_haversine_meters",
    "gps_number_of_stay_points",
    "gps_total_time_at_clusters_seconds",
    "gps_location_entropy",
    "gps_location_variance",
    "gps_radius_of_gyration_meters",
]


def simulate_gps_features(home_lat, home_lon, mobility_level, awake_minutes):
    """Simulate one participant-day's GPS itinerary and derive trip stats from it."""
    k = int(np.clip(RNG.poisson(2 + 10 * mobility_level), 1, 20))

    spread_deg = 0.0003 + 0.006 * mobility_level
    lat_pts = np.concatenate([[home_lat], home_lat + RNG.normal(0, spread_deg, k)])
    lon_pts = np.concatenate([[home_lon], home_lon + RNG.normal(0, spread_deg, k)])

    # dwell-time share at each location (including home)
    dwell_frac = RNG.dirichlet(np.ones(k + 1))
    time_at_clusters_seconds = 0.75 * awake_minutes * 60 * RNG.uniform(0.8, 1.0)

    entropy = -np.sum(dwell_frac * np.log2(dwell_frac + 1e-12))

    centroid_lat = np.average(lat_pts, weights=dwell_frac)
    centroid_lon = np.average(lon_pts, weights=dwell_frac)
    dists_from_centroid = haversine_meters(lat_pts, lon_pts, centroid_lat, centroid_lon)

    radius_of_gyration = np.sqrt(np.sum(dwell_frac * dists_from_centroid ** 2))
    location_variance = np.average(dists_from_centroid ** 2, weights=dwell_frac)

    # path length visiting points in order, home -> stop1 -> ... -> home
    loop_lat = np.concatenate([lat_pts, [home_lat]])
    loop_lon = np.concatenate([lon_pts, [home_lon]])
    leg_dists = haversine_meters(loop_lat[:-1], loop_lon[:-1], loop_lat[1:], loop_lon[1:])
    total_haversine = np.sum(leg_dists)

    return {
        "gps_total_haversine_meters": np.clip(total_haversine, 0, 20000),
        "gps_number_of_stay_points": k,
        "gps_total_time_at_clusters_seconds": np.clip(time_at_clusters_seconds, 0, 100000),
        "gps_location_entropy": np.clip(entropy, 0, 10),
        "gps_location_variance": np.clip(location_variance, 0, 500000),
        "gps_radius_of_gyration_meters": np.clip(radius_of_gyration, 0, 20000),
    }

Overwriting gps_features.py


In [13]:
%%writefile gps_features.py
import numpy as np

from config import RNG
from geo import haversine_meters

GPS_COLS = [
    "gps_total_haversine_meters",
    "gps_number_of_stay_points",
    "gps_total_time_at_clusters_seconds",
    "gps_location_entropy",
    "gps_location_variance",
    "gps_radius_of_gyration_meters",
]


def simulate_gps_features(home_lat, home_lon, mobility_level, awake_minutes):
    """Simulate one participant-day's GPS itinerary and derive trip stats from it."""
    k = int(np.clip(RNG.poisson(2 + 10 * mobility_level), 1, 20))

    spread_deg = 0.0003 + 0.006 * mobility_level
    lat_pts = np.concatenate([[home_lat], home_lat + RNG.normal(0, spread_deg, k)])
    lon_pts = np.concatenate([[home_lon], home_lon + RNG.normal(0, spread_deg, k)])

    # dwell-time share at each location (including home)
    dwell_frac = RNG.dirichlet(np.ones(k + 1))
    time_at_clusters_seconds = 0.75 * awake_minutes * 60 * RNG.uniform(0.8, 1.0)

    entropy = -np.sum(dwell_frac * np.log2(dwell_frac + 1e-12))

    centroid_lat = np.average(lat_pts, weights=dwell_frac)
    centroid_lon = np.average(lon_pts, weights=dwell_frac)
    dists_from_centroid = haversine_meters(lat_pts, lon_pts, centroid_lat, centroid_lon)

    radius_of_gyration = np.sqrt(np.sum(dwell_frac * dists_from_centroid ** 2))
    location_variance = np.average(dists_from_centroid ** 2, weights=dwell_frac)

    # path length visiting points in order, home -> stop1 -> ... -> home
    loop_lat = np.concatenate([lat_pts, [home_lat]])
    loop_lon = np.concatenate([lon_pts, [home_lon]])
    leg_dists = haversine_meters(loop_lat[:-1], loop_lon[:-1], loop_lat[1:], loop_lon[1:])
    total_haversine = np.sum(leg_dists)

    return {
        "gps_total_haversine_meters": np.clip(total_haversine, 0, 20000),
        "gps_number_of_stay_points": k,
        "gps_total_time_at_clusters_seconds": np.clip(time_at_clusters_seconds, 0, 100000),
        "gps_location_entropy": np.clip(entropy, 0, 10),
        "gps_location_variance": np.clip(location_variance, 0, 500000),
        "gps_radius_of_gyration_meters": np.clip(radius_of_gyration, 0, 20000),
    }

Overwriting gps_features.py


In [14]:
%%writefile sleep_features.py
import numpy as np

from config import NUM_PEOPLE, RNG

SLEEP_COLS = [
    "sleep_total_sleep_minutes",
    "sleep_avg_sleep_hours",
    "sleep_total_night_screen_minutes",
    "sleep_total_screen_interruptions",
]


def simulate_sleep_features(persona, mood_deficit):
    duration_hours = np.clip(
        7.5 - 1.5 * mood_deficit + RNG.normal(0, 0.7, NUM_PEOPLE), 4, 10
    )
    total_sleep_minutes = duration_hours * 60

    night_screen_minutes = np.clip(
        15 + 70 * mood_deficit * persona["screen_habit"] + RNG.normal(0, 10, NUM_PEOPLE),
        0, 300,
    )
    interruptions = np.clip(
        RNG.poisson(1 + 4 * mood_deficit * persona["screen_habit"]), 0, 10
    )

    return {
        "sleep_total_sleep_minutes": total_sleep_minutes,
        "sleep_avg_sleep_hours": duration_hours,
        "sleep_total_night_screen_minutes": night_screen_minutes,
        "sleep_total_screen_interruptions": interruptions,
    }

Overwriting sleep_features.py


In [15]:
%%writefile activity_features.py
import numpy as np

from config import NUM_PEOPLE, RNG

ACTIVITY_COLS = [
    "acc_vigorous_pa_minutes",
    "acc_non_vigorous_pa_minutes",
    "acc_sedentary_minutes",
    "acc_total_active_minutes",
    "acc_mean_activity_intensity",
    "acc_max_activity_intensity",
    "acc_activity_intensity_std",
    "acc_activity_bout_count",
    "acc_longest_sedentary_bout_minutes",
    "acc_activity_fragmentation_index",
    "acc_morning_activity_minutes",
    "acc_afternoon_activity_minutes",
    "acc_evening_activity_minutes",
    "acc_night_activity_minutes",
    "acc_accelerometer_sample_count",
    "acc_data_coverage_hours",
]


def simulate_activity_features(persona, mood_deficit, awake_minutes, chronotype_offset):
    activity_level = persona["activity_level"].to_numpy()

    vigorous = np.clip(activity_level * 35 + RNG.normal(0, 8, NUM_PEOPLE), 0, 180)
    non_vigorous = np.clip(
        activity_level * 70 + (1 - mood_deficit) * 30 + RNG.normal(0, 15, NUM_PEOPLE), 0, 200
    )
    total_active = vigorous + non_vigorous
    sedentary = np.clip(awake_minutes - total_active, 0, 1200)

    mean_intensity = np.clip(0.5 + 2.2 * activity_level + RNG.normal(0, 0.3, NUM_PEOPLE), 0, 5)
    max_intensity = np.clip(mean_intensity * 1.6 + RNG.normal(0, 0.4, NUM_PEOPLE), 0, 10)
    intensity_std = np.clip(0.3 + 1.4 * (1 - activity_level) + RNG.normal(0, 0.2, NUM_PEOPLE), 0, 5)

    bout_count = np.clip(RNG.poisson(4 + 14 * activity_level), 0, 100)
    longest_sedentary_bout = np.clip(
        30 + (1 - activity_level) * 280 + mood_deficit * 150 + RNG.normal(0, 30, NUM_PEOPLE), 0, 600
    )
    fragmentation_index = np.clip(bout_count / (total_active + 1), 0, 1)

    morning, afternoon, evening, night = _split_activity_by_time_of_day(
        total_active, mood_deficit, chronotype_offset
    )

    sample_count = np.clip((awake_minutes / 60) * RNG.uniform(700, 1300, NUM_PEOPLE), 5000, 30000)
    coverage_hours = np.clip((awake_minutes / 60) * RNG.uniform(0.75, 1.0, NUM_PEOPLE), 1, 24)

    return {
        "acc_vigorous_pa_minutes": vigorous,
        "acc_non_vigorous_pa_minutes": non_vigorous,
        "acc_sedentary_minutes": sedentary,
        "acc_total_active_minutes": total_active,
        "acc_mean_activity_intensity": mean_intensity,
        "acc_max_activity_intensity": max_intensity,
        "acc_activity_intensity_std": intensity_std,
        "acc_activity_bout_count": bout_count,
        "acc_longest_sedentary_bout_minutes": longest_sedentary_bout,
        "acc_activity_fragmentation_index": fragmentation_index,
        "acc_morning_activity_minutes": morning,
        "acc_afternoon_activity_minutes": afternoon,
        "acc_evening_activity_minutes": evening,
        "acc_night_activity_minutes": night,
        "acc_accelerometer_sample_count": sample_count,
        "acc_data_coverage_hours": coverage_hours,
    }


def _split_activity_by_time_of_day(total_active, mood_deficit, chronotype_offset):
    """Spread total active minutes across morning/afternoon/evening/night,
    shifted earlier or later depending on each participant's chronotype."""
    morning_bias = np.clip(0.30 - 0.05 * chronotype_offset, 0.05, 0.6)
    weights = np.stack([
        morning_bias,
        np.full(NUM_PEOPLE, 0.30),
        np.full(NUM_PEOPLE, 0.25),
        np.clip(0.15 + mood_deficit * 0.1, 0.05, 0.4),
    ], axis=1)
    weights = weights / weights.sum(axis=1, keepdims=True)

    split = weights * total_active[:, None]
    return np.clip(split, 0, 120).T

Overwriting activity_features.py


In [16]:
%%writefile screen_features.py
import numpy as np

from config import NUM_PEOPLE, RNG

SCREEN_COLS = [
    "screen_total_screen_time_in_seconds",
    "screen_num_of_events_total",
    "screen_morning_screen_time_in_seconds",
    "screen_num_of_events_morning",
    "screen_afternoon_screen_time_in_seconds",
    "screen_num_of_events_afternoon",
    "screen_evening_screen_time_in_seconds",
    "screen_num_of_events_evening",
    "screen_nighttime_screen_time_in_seconds",
    "screen_num_of_events_nighttime",
]


def simulate_screen_features(persona, mood_deficit, chronotype_offset):
    screen_habit = persona["screen_habit"].to_numpy()

    total_seconds = np.clip(
        3600 * (1 + 4 * screen_habit + 2.5 * mood_deficit) + RNG.normal(0, 600, NUM_PEOPLE),
        0, 20000,
    )
    total_events = np.clip(RNG.poisson(20 + 150 * screen_habit), 0, 300)

    time_split, event_split = _split_screen_by_time_of_day(
        total_seconds, total_events, mood_deficit, chronotype_offset
    )

    morning_t, afternoon_t, evening_t, night_t = time_split
    morning_e, afternoon_e, evening_e, night_e = event_split

    return {
        "screen_total_screen_time_in_seconds": total_seconds,
        "screen_num_of_events_total": total_events,
        "screen_morning_screen_time_in_seconds": morning_t,
        "screen_num_of_events_morning": morning_e,
        "screen_afternoon_screen_time_in_seconds": afternoon_t,
        "screen_num_of_events_afternoon": afternoon_e,
        "screen_evening_screen_time_in_seconds": evening_t,
        "screen_num_of_events_evening": evening_e,
        "screen_nighttime_screen_time_in_seconds": night_t,
        "screen_num_of_events_nighttime": night_e,
    }


def _split_screen_by_time_of_day(total_seconds, total_events, mood_deficit, chronotype_offset):
    night_bias = np.clip(0.10 + 0.20 * mood_deficit, 0.05, 0.5)
    weights = np.stack([
        np.clip(0.25 - 0.04 * chronotype_offset, 0.05, 0.5),
        np.full(NUM_PEOPLE, 0.30),
        np.full(NUM_PEOPLE, 0.25),
        night_bias,
    ], axis=1)
    weights = weights / weights.sum(axis=1, keepdims=True)

    time_split = np.clip(weights * total_seconds[:, None], 0, 8000).T
    event_split = np.clip(weights * total_events[:, None], 0, 120).T

    return time_split, event_split

Overwriting screen_features.py


In [17]:
%%writefile combine.py
import numpy as np
import pandas as pd

from config import NUM_PEOPLE, DEMOGRAPHIC_COLS
from gps_features import GPS_COLS, simulate_gps_features
from sleep_features import SLEEP_COLS, simulate_sleep_features
from activity_features import ACTIVITY_COLS, simulate_activity_features
from screen_features import SCREEN_COLS, simulate_screen_features

# the full behavioral feature set, assembled from each domain's own columns
BEHAVIOR_COLS = GPS_COLS + SLEEP_COLS + ACTIVITY_COLS + SCREEN_COLS

FINAL_COLS = (
    ["participantid", "date_local"]
    + BEHAVIOR_COLS
    + DEMOGRAPHIC_COLS
    + ["daily_sum", "daily_average_score"]
)


def simulate_one_day(persona, mood_deficit, chronotype_offset):
    """Simulate every feature domain for one day, then combine them into a
    single per-participant row."""
    sleep = simulate_sleep_features(persona, mood_deficit)
    awake_minutes = 1440 - sleep["sleep_total_sleep_minutes"]

    gps_rows = [
        simulate_gps_features(
            persona["home_lat"][i], persona["home_lon"][i],
            persona["mobility_level"][i], awake_minutes[i],
        )
        for i in range(NUM_PEOPLE)
    ]
    gps = pd.DataFrame(gps_rows).to_dict(orient="list")

    activity = simulate_activity_features(persona, mood_deficit, awake_minutes, chronotype_offset)
    screen = simulate_screen_features(persona, mood_deficit, chronotype_offset)

    day_df = pd.DataFrame({**gps, **sleep, **activity, **screen})

    # composite scores, derived from the combined day rather than sampled directly
    day_df["daily_sum"] = np.clip(
        day_df["acc_total_active_minutes"]
        + day_df["screen_total_screen_time_in_seconds"] / 60
        + day_df["sleep_total_sleep_minutes"] / 2,
        0, 2000,
    )
    day_df["daily_average_score"] = np.clip(day_df["daily_sum"] / 20, 0, 100)

    return day_df[BEHAVIOR_COLS + ["daily_sum", "daily_average_score"]]


def combine_with_demographics(behavioral_df, demographics_df, num_days):
    """Repeat each participant's fixed demographics across their daily rows
    and stitch everything into the final, ordered dataset."""
    demo_repeated = demographics_df.loc[demographics_df.index.repeat(num_days)].reset_index(drop=True)
    dataset = pd.concat([behavioral_df, demo_repeated], axis=1)
    return dataset[FINAL_COLS]

Writing combine.py


In [18]:
%%writefile generate_synthetic_data.py
from datetime import timedelta

import numpy as np
import pandas as pd

from config import NUM_PEOPLE, NUM_DAYS, START_DATE
from personas import make_personas, make_demographics
from combine import simulate_one_day, combine_with_demographics


def main():
    persona = make_personas()
    demographics = make_demographics()

    mood_deficit = demographics["depression_score"].to_numpy() / 60.0
    chronotype_offset = persona["chronotype_offset"].to_numpy()

    dates = [START_DATE + timedelta(days=t) for t in range(NUM_DAYS)]

    all_days = []
    for t in range(NUM_DAYS):
        day_df = simulate_one_day(persona, mood_deficit, chronotype_offset)
        day_df.insert(0, "date_local", dates[t].isoformat())
        day_df.insert(0, "participantid", np.arange(NUM_PEOPLE))
        all_days.append(day_df)

    behavioral = (
        pd.concat(all_days, ignore_index=True)
        .sort_values(["participantid", "date_local"])
        .reset_index(drop=True)
    )

    dataset = combine_with_demographics(behavioral, demographics, NUM_DAYS)

    dataset.to_csv("synthetic_diffusion_dataset.csv", index=False)

    print(dataset.shape)
    print(dataset.head())


if __name__ == "__main__":
    main()

Writing generate_synthetic_data.py


In [19]:
!python generate_synthetic_data.py


(14000, 49)
   participantid  date_local  ...   daily_sum  daily_average_score
0              0  2024-01-01  ...  575.085274            28.754264
1              0  2024-01-02  ...  611.589488            30.579474
2              0  2024-01-03  ...  640.135062            32.006753
3              0  2024-01-04  ...  565.632189            28.281609
4              0  2024-01-05  ...  538.299067            26.914953

[5 rows x 49 columns]


In [20]:
import pandas as pd
df = pd.read_csv("synthetic_diffusion_dataset.csv")
df.head()

,participantid,date_local,gps_total_haversine_meters,gps_number_of_stay_points,gps_total_time_at_clusters_seconds,gps_location_entropy,gps_location_variance,gps_radius_of_gyration_meters,sleep_total_sleep_minutes,sleep_avg_sleep_hours,...,age,biological_sex,identifying_gender,phone,mental_health_diagnosis,disorder_name___10,disorder_name___1,depression_score,daily_sum,daily_average_score
0,0,2024-01-01,1157.601474,5,38836.105967,2.026419,7463.617160,86.392229,375.110334,6.251839,...,35,Female,Woman,iOS,1,0,0,32,575.085274,28.754264
1,0,2024-01-02,2973.124105,5,40972.553117,2.038659,54594.525511,233.654714,391.817985,6.530300,...,35,Female,Woman,iOS,1,0,0,32,611.589488,30.579474
2,0,2024-01-03,1519.862698,3,39698.043859,1.639394,53882.308120,232.125630,486.657019,8.110950,...,35,Female,Woman,iOS,1,0,0,32,640.135062,32.006753
3,0,2024-01-04,1657.660068,4,41733.347764,2.109996,28102.092181,167.636786,403.093839,6.718231,...,35,Female,Woman,iOS,1,0,0,32,565.632189,28.281609
4,0,2024-01-05,1429.898673,5,43902.930465,2.468094,37155.578900,192.757824,348.631142,5.810519,...,35,Female,Woman,iOS,1,0,0,32,538.299067,26.914953
